# Week 3 Day 4: Models

---
Until now we used `pipeline()` which handled everything in one line.
Today we skip the pipeline and talk **directly to the model**.

The full flow we will do manually:
```
Text  →  Tokenizer  →  Token IDs  →  Model  →  Token IDs  →  Tokenizer  →  Text
```
In this notebook, we will work with five instruction-tuned Large Language Models
from Hugging Face:

| Variable | Model | Company | Parameters |
|---|---|---|---:|
| `LLAMA` | Llama-3.2-1B-Instruct | Meta | 1B |
| `PHI` | Phi-4-mini-instruct | Microsoft | ~4B |
| `GEMMA` | Gemma-3-270m-it | Google | 270M |
| `QWEN` | Qwen3-4B-Instruct | Alibaba | 4B |
| `DEEPSEEK` | DeepSeek-R1-Distill-Qwen-1.5B | DeepSeek AI | 1.5B |

All five models are **instruction-tuned models**, meaning they are designed
to follow instructions and generate conversational responses.

## Step 1 : Install Libraries

We need:
- **`transformers`**: The main HuggingFace library with all Auto classes and models.
- **`accelerate`**: Helps load large models efficiently across CPU/GPU.
- **`bitsandbytes`**: Enables model quantization (loading big models in less memory).

In [1]:
!pip install -qU transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.5 MB/s eta 0:00:00


## Step 2 — Imports

- `AutoTokenizer` → loads the right tokenizer for any model
- `AutoModelForCausalLM` → loads the language model
- `TextStreamer` → streams generated text as it is produced
- `BitsAndBytesConfig` → configures 4-bit quantization
- `torch` → PyTorch deep-learning framework
- `gc` → Python garbage collector — used to free GPU memory between models

In [2]:
from huggingface_hub import login
from google.colab import userdata
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
)
import torch
import gc

## Step 3 — Login to Hugging Face

In [3]:
hf_token = userdata.get('HF_TOKEN')

login(hf_token, add_to_git_credential=True)

print("Logged in to HuggingFace successfully!")

Logged in to HuggingFace successfully!


## Step 4 — GPU and Memory Check

We will use a GPU to run the language models.

Google Colab's T4 GPU provides approximately **16 GB of VRAM**.

Larger models can require several GB of memory even before accounting for
the tokenizer, activations, KV cache, and other runtime requirements.

Therefore, we will use **4-bit quantization** for the larger models.

In [4]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU Memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


## Step 5 — Understanding Model Size

A model's size is commonly described by the number of parameters.

For example:

**8B = 8 billion parameters**

Each parameter requires memory depending on the numerical precision used to
store it.

| Precision | Bytes / Parameter | Memory for 8B Parameters |
|---|---:|---:|
| FP32 | 4 bytes | ~32 GB |
| FP16 / BF16 | 2 bytes | ~16 GB |
| INT8 | 1 byte | ~8 GB |
| INT4 | 0.5 byte | ~4 GB |

These are approximate **weight-memory** calculations. Actual GPU usage is higher because inference also needs memory for activations, KV cache, temporary tensors, and other runtime overhead.

This is why quantization is useful: it allows larger models to fit on smaller GPUs.

In [5]:
parameters = 8_000_000_000

print("Memory requirements for an 8B parameter model:")
print("-" * 45)

print(f"float32 (4 bytes/param): {parameters * 4 / 1e9:.1f} GB")
print(f"float16 (2 bytes/param): {parameters * 2 / 1e9:.1f} GB")
print(f"int8    (1 byte/param):  {parameters * 1 / 1e9:.1f} GB")
print(f"int4    (0.5 byte/param): {parameters * 0.5 / 1e9:.1f} GB")

print()
print("Actual GPU memory usage will be higher than weight memory.")

Memory requirements for an 8B parameter model:
---------------------------------------------
float32 (4 bytes/param): 32.0 GB
float16 (2 bytes/param): 16.0 GB
int8    (1 byte/param):  8.0 GB
int4    (0.5 byte/param): 4.0 GB

Actual GPU memory usage will be higher than weight memory.


## Step 6 — Define the 5 Models We Will Use

In [6]:
# Define the 5 Models We Will Use
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

## Step 7 — Define the Prompt

In [7]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

## Step 8 — Configure 4-Bit Quantization

A 4-bit quantization configuration allows model weights to be stored using
approximately 4 bits per parameter instead of 16 bits.

We will use **BitsAndBytes** to configure 4-bit loading.

### Configuration

- `load_in_4bit=True` → loads weights in 4-bit precision
- `bnb_4bit_use_double_quant=True` → further reduces quantization overhead
- `bnb_4bit_compute_dtype` → controls the dtype used during computation
- `bnb_4bit_quant_type='nf4'` → uses the NF4 quantization format

In [8]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

print("4-bit quantization configured successfully.")

4-bit quantization configured successfully.


## Step 9 — Tokenize the Input

This is the first step of running a model manually.

- `AutoTokenizer.from_pretrained(LLAMA)` → downloads and loads the Llama tokenizer
- `tokenizer.pad_token = tokenizer.eos_token` → Llama has no pad token by default; we reuse the end-of-sequence token to avoid a warning
- `apply_chat_template(messages, ...)` → converts our `messages` list into Llama's exact expected format with all the special tokens (`<|start_header_id|>`, `<|eot_id|>`, etc.)
- `return_tensors="pt"` → return as a PyTorch tensor (not a plain Python list)
- `.to("cuda")` → move the tensor to the GPU so the model can use it

In [9]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
input = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True )
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

### What does the tokenized input look like?

`inputs` tensor — you'll see a list of integers on a CUDA tensor.
Each number is a token ID representing one piece of the prompt (including all chat formatting).

In [10]:
input

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Oct 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

In [11]:
inputs

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2371,   5020,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
            315,   2956,  57116, 128009]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

## Step 10 — Load the Model

Now we download and load the actual neural network weights:

- `AutoModelForCausalLM.from_pretrained(LLAMA)` → downloads the model from HuggingFace Hub
- `device_map="auto"` → automatically places layers on GPU (or CPU if GPU runs out of memory)
- `quantization_config=quant_config` → applies 4-bit compression as the weights are loaded

In [12]:
# The model

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quantization_config)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

### How much GPU memory does the model use?

`get_memory_footprint()` returns bytes → we divide by `1e6` to get **megabytes**.

For Llama-3.2-1B in 4-bit you should see roughly **~700–900 MB**.
Without quantization (float16) it would be ~2 GB — quantization made it ~2.5× smaller.

In [13]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 1,012.0 MB


## Step 11 — Look Inside the Model (The Transformer Architecture)

Just printing the `model` object shows you the **entire neural network structure**.

Look for these key parts in the output:

| Layer | What it does |
|---|---|
| `embed_tokens` | Converts each token ID into a vector of numbers (the "meaning" of each token) |
| `layers` (repeated 16x) | The 16 Transformer decoder blocks — where the actual thinking happens |
| `self_attn` (inside each layer) | **Attention**: each token looks at all others and decides what's relevant |
| `mlp` (inside each layer) | **Feed-forward network**: processes the attention output |
| `norm` | Normalization: keeps numbers stable so training doesn't explode |
| `lm_head` | Final layer: converts the last vector → probabilities over the full vocabulary |

You'll also see `Linear4bit` — confirming 4-bit quantization is active on the weight matrices.

In [14]:
# Execute this cell and look at what gets printed; investigate the layers
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

## Step 12 — Generate Output (Raw Token IDs)

`model.generate()` is the **core method** for text generation.

- `inputs` → the token IDs of our prompt (from Step 7)
- `max_new_tokens=80` → generate at most 80 new tokens after the prompt

The output is a **tensor of token IDs** — the model's raw output. Not readable yet.
`outputs[0]` takes the first item (we only have one prompt in the batch).

In [15]:
# OK, with that, now let's run the model!

# Unpack the inputs dictionary (passing both input_ids and attention_mask)
outputs = model.generate(**inputs, max_new_tokens=80)
outputs[0]

tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   2371,   5020,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
           315,   2956,  57116, 128009, 128006,  78191, 128007,    271,   8586,
           596,    264,  22380,    430,    596,    704,    315,    420,   1917,
          1473,  10445,   1550,    279,    828,  28568,   4546,    264,  36865,
           311,    279,   6574,   1980,  18433,    814,   4934,    311,   1935,
           872,   6492,    311,    264,   5190,   2237,    382,  38257,  22380,
         11335,    389,    279,  17571,    330,  90447,   2237,      1,    304,
           264,  11156,   5647,     11,    719,   1101,  15407,    279,  24016,
          1180,    315,  19261,    369,    264,  36865,     13,    358,   3987,
           279,    828,  14248,    304, 

## Step 13 — Decode the Output Back to Text

`tokenizer.decode()` converts the token IDs → a readable string.

Notice: the output includes **the full prompt + the generated response** because `model.generate()` returns the entire sequence (input tokens + new tokens combined).

In [16]:
# Well that doesn't make much sense!
# How about this..
tokenizer.decode(outputs[0])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Oct 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere\'s a joke that\'s out of this world:\n\nWhy did the data scientist bring a ladder to the meeting?\n\nBecause they wanted to take their analysis to a higher level.\n\n(This joke plays on the phrase "higher level" in a technical sense, but also references the literal act of reaching for a ladder. I hope the data scientists in the room can "process" the humor'

## Step 14 — Free GPU Memory

Before loading the next model, we must delete the current one and release GPU memory.

Three steps are needed:
1. `del model, inputs, tokenizer, outputs` → removes Python's references to these objects
2. `gc.collect()` → Python's garbage collector frees CPU/system RAM
3. `torch.cuda.empty_cache()` → PyTorch releases GPU VRAM back to the OS

Skip any one of these and memory may not be freed properly.

In [17]:
# Clean up memory

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## Step 15 — A Reusable `generate()` Function with Streaming

Instead of repeating Steps 7–12 for each model, we wrap everything into one clean function.

Two new things compared to before:

**1. `TextStreamer`** — prints tokens as they are generated (streaming output, like ChatGPT)  
Instead of waiting for all 80 tokens and then printing, you see each word appear in real time.

**2. `add_generation_prompt=True`** in `apply_chat_template`  
This adds the `<|start_header_id|>assistant<|end_header_id|>` tokens at the end of the prompt.  
Without it, the model might just continue the user's message instead of giving a reply.

**`quant=True/False`** lets us skip quantization for tiny models (like Gemma-270M) that already fit in VRAM.

In [25]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant = True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quantization_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(**input, max_new_tokens=max_new_tokens)
  return tokenizer.decode(outputs[0])


## Step 16 — Run Phi-4 Mini (Microsoft)

`quant=True` (default) → loads with 4-bit quantization.  
Watch the output **stream in word by word** — that's `TextStreamer` working!

The function handles everything: load tokenizer → format prompt → load model → generate → stream output.

In [19]:
generate(PHI, messages)

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

"<|user|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:\n\nWhy did the data scientist break up with the computer?\n\nBecause it kept asking for more space!<|end|>"

## Step 17 — Run Gemma-270M (Google)

Notice `quant=False` here.  
Gemma-3-270m is only **270 million parameters** — tiny enough to load in full precision without quantization.

**Rule of thumb:** models under ~500M params → `quant=False`. Models over 1B → `quant=True`.

In [21]:
generate(GEMMA, messages, quant=False)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  536MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

'<bos><start_of_turn>user\nTell a joke for a room of Data Scientists<end_of_turn>\n<start_of_turn>model\nWhy did the data scientist get fired? \nBecause he kept trying to make the numbers **bigger**! \n<end_of_turn>'

## Step 18 — Run Qwen3 (Alibaba)

4B parameters → needs quantization → `quant=True` (default).  
Compare the style and quality of Qwen's response vs Phi and Gemma!

In [22]:
generate(QWEN, messages)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

"<|im_start|>user\nTell a joke for a room of Data Scientists<|im_end|>\n<|im_start|>assistant\nSure! Here's a *data science* pun joke, specially crafted for a room full of statisticians, ML engineers, and anyone who loves regression:\n\n---\n\nWhy did the data scientist break up with their partner?\n\nBecause they were *not* a good fit for the model — there was *no significant correlation* in the relationship, and the residuals were just too high.\n\nBonus points if you noticed"

## Step 19 — Run DeepSeek-R1 (Reasoning Model)

This is different from all the others — DeepSeek-R1 is a **reasoning model**.

Normal LLMs answer immediately. Reasoning models **think step by step first** (like OpenAI's o1/o3), then give the answer.

- `quant=False` → the 1.5B distilled version is small enough without quantization
- `max_new_tokens=500` → reasoning models need many more tokens because they write out their thinking process before the final answer

Look for `<think>...</think>` tags in the output — that's the model's internal reasoning showing!

In [27]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

'<｜begin▁of▁sentence｜><｜User｜>Tell a joke for a room of Data Scientists<｜Assistant｜><think>\nAlright, I need to come up with a joke for a room filled with data scientists. Let me think about what they might be thinking or discussing. They probably love statistics, data, and maybe a bit of humor. \n\nHmm, maybe something related to their field. Data scientists love numbers and data, so perhaps a joke about numbers or math. Or maybe a pun involving words they\'re familiar with.\n\nI should make it light and funny, something that would make them laugh. Maybe a riddle or a play on words. Let me brainstorm a few ideas.\n\nFirst, I can think of a common phrase or pun. Maybe something like "Why don\'t scientists go to parties?" That\'s a classic one, but maybe I can twist it. "Why don\'t scientists go to parties? Because they have data to analyze!"\n\nWait, that\'s a play on words. "Data" is a plural noun, so "data" could mean something else. But in this context, it\'s clear. Maybe I can make